# Validation of flatcor's three default priors

This notebook presents the validation of [flatcor](https://github.com/dm-k1/flatcor) (3.0, in
development), which offers three default priors for Bayesian linear regression: flat partial *r*,
flat *R* and flat *R*², each a Beta(*a*, 1) prior on *R*² with a uniform direction. The validation
is what someone needs to trust that each prior works within the Bayesian framework. Three things are
validated for every prior, each in two ways (the table in
[`docs/working/REDESIGN.md`](docs/working/REDESIGN.md)):

| | 1. Prior sampling | 2. Posterior estimation | 3. Bayes factors |
|---|---|---|---|
| **a. Computed correctly?** | 1a: the prior draws against the exact prior | 2a: simulation-based calibration | 3a: calibration of the Bayes factor; bridge sampling's error |
| **b. Behaves like a default?** | 1b: invariance | 2b: estimation against a known truth | 3b: the Bayarri et al. (2012) criteria |

flatcor does all three with one Stan model; its Bayes factors estimate each model's marginal
likelihood by bridge sampling. Every check is a standard one, the same for the three priors.

Each study is a folder in `R/2_studies/` (`design.R`, `code.R`), run by the pipeline
[`_targets.R`](_targets.R), that saves its tables to `results/studies/study_<cell>.rds`; the scripts in
`R/3_figures/` make the figures from those files; this notebook only presents them. What each study
tests, expects and why is in [`docs/validation_tests.md`](docs/validation_tests.md). **Results are estimates against
what is expected:** every quantity is shown with its expected value (fixed in advance), its Monte
Carlo standard error (MCSE) and the distance between them in MCSEs, and under each table how many
lie beyond 2 and 3 MCSEs against what chance gives. There is no pass/fail per row.

Each study shows the tables that decide its result, its figures and, in a line, what the result
is. The rest (every row, the counts beyond 2 and 3 MCSEs, divergences, iterations) is under
**Details** at the end.

**This notebook only presents.** It reads the results files and shows each study's tables and
figures; it runs nothing. The studies run with `Rscript run_all.R`, on a machine with many cores
(the full validation is far too big for a notebook; [`cluster/README.md`](cluster/README.md)
describes a run on a cluster or AWS). Figures that are not there yet (after downloading a run's results, say)
are drawn here with `make_figures.R`. With `smoke <- TRUE` the notebook shows the smoke test
instead: a corner of each design and a handful of datasets, to test the code. Its results and
figures are in `cache/`, and its numbers mean nothing.

In [ ]:
smoke   <- FALSE       # TRUE: the smoke test's results instead of the full studies
studies <- c("1a", "1b", "2a", "2b", "3a", "3b")   # the studies to show

In [ ]:
## Jupyter starts in the notebook's folder, the project folder, where the scripts run too
source("R/4_notebook/present.R")   # show_study(), show_table()
show_table(overview_of(studies, smoke), if (smoke) "Overview (smoke test)" else "Overview")

---

# 1. Prior sampling

## 1a. Does the software draw the prior it claims?

flatcor's Stan model samples *R*, with *R*² ~ Beta(*a*, 1), and a uniform direction, and computes the
coefficients from them. With the data switched off it should draw exactly the prior: the quantiles of
*r*² (one predictor) and of *R*² (three and eight predictors) against those of Beta(*a*, 1), and of
each partial correlation against the prior as a recipe (`draw_coefficients_from_prior()`), with
Monte Carlo SEs that allow for autocorrelated draws.

In [ ]:
show_study("1a", smoke = smoke, tables = list(
  "KS distance to the expected distribution, against its critical value at .01" = "ks"
))

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): the Stan model draws the prior it claims: all 18 KS distances are below the critical
value, and none of the 90 percentiles lies beyond 3 MCSE.

## 1b. Is the Bayes factor invariant to the units and coding of the predictors?

Rescaling, shifting or recoding the predictors and the outcome should leave every Bayes factor
unchanged (Bayarri et al., 2012): the change in log Bayes factor, expected 0, against the Monte Carlo
error of bridge sampling.

In [ ]:
show_study("1b", smoke = smoke, tables = list(
  "Mean change in the log Bayes factor, per prior" = "pooled"
))

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): invariant: the pooled changes in log BF are at most .006, against Δ = .095. The excess per
dataset is bridge sampling's optimistic error (×1.29 pooled, about 1.4 in most cells; study 3a).

---

# 2. Posterior estimation

## 2a. Is the posterior computed correctly?

Simulation-based calibration (Talts et al., 2018): parameters drawn from each prior, data from them,
the model fitted; if the posterior is right, the rank of each true value among its posterior draws is
uniform. The mean and SD of the scaled ranks and a chi-square over 20 bins against their values under
uniformity, and Stan's divergent transitions.

In [ ]:
show_study("2a", smoke = smoke,
  tables = list(
    "Ranks of the true values against a uniform distribution (mean and SD of the scaled rank)" =
      function(tables) subset(as.data.frame(tables$ranks), !grepl("chi-square", quantity))
  ),
  notes = list(function(tables) {
    divergence_sentence(tables$divergences, "fits", "fits_with_divergences")
  })
)

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): simulation-based calibration is fine. One estimate in 54 lies beyond 3 MCSE (flat *R*,
*k* = 8, SD of *R*²'s rank .328; 1.9 MCSE in an earlier run). σ's mean rank of .448 at *k* = 3 is
one shared event: the priors share σ and the noise.

## 2b. How close are the posterior estimates to a known truth?

On data with known slopes (slope of *x*₁ = 0.2): the bias of *x*₁'s posterior mean, the
coverage of its 95% interval, and the prediction error of the posterior means relative to OLS, for
the three priors, JZS, Zellner–Siow and OLS. The references (bias 0, coverage .95, prediction ratio 1)
are not guaranteed for a Bayesian estimate, so this describes how the priors behave.

In [ ]:
show_study("2b", smoke = smoke,
  tables = list("Per method: the mean and range over the cells" = summary_2b),
  notes = list(function(tables) {
    divergence_sentence(tables$divergent, "fits", "fits_with_divergences")
  })
)

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): flat partial *r* behaves like Zellner–Siow (bias −6%, coverage .948). Flat *R* and flat
*R*² shrink hard at *k* = 10 (bias down to −59% and −48%, coverage down to .81 and .88) and predict
best there (0.39–0.42 of OLS's excess error).

---

# 3. Bayes factors

## 3a. Are the Bayes factors right?

Calibration (Schad et al., 2023): on data from each prior, half from the model without *x*₁, the
posterior probability of the model without *x*₁, averaged over the datasets, should equal its prior
probability, ½ (the data-averaged posterior, which decides). Described per bin, where the share
without *x*₁ among datasets with a given BF₁₀ should equal 1 / (1 + BF₁₀), and as a logistic
calibration line (intercept 0, slope 1). And bridge sampling's error: the spread of the same Bayes
factor over five runs with fresh posterior draws against the error bridgesampling reports (expected
ratio 1); the factor by which it falls short corrects the errors of 1b and 3b.1–3b.2.

In [ ]:
show_study("3a", smoke = smoke, tables = list(
  "Data-averaged posterior per prior and cell (decides part A)" = "data_averaged_posterior",
  "Bridge sampling's error from repeated runs" = "repeated_runs",
  "Bridge sampling's error: the shortfall factor used by 1b and 3b" = "bridge_shortfall"
))

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): calibrated: the data-averaged posterior is .498–.523 against ½, all within 1 MCSE.
Bridge sampling's real error is 1.3–1.4 times its reported one, negligible in size (about .005
on the log BF).

## 3b. Do the Bayes factors meet the default-prior criteria?

The criteria of Bayarri, Berger, Forte and García-Donato (2012): predictive matching, information
consistency, model-selection consistency for a fixed and a growing number of predictors (*k* at most
90), and intrinsic consistency; invariance is 1b. Where the expectation is a number (predictive
matching, the asymptotic rates) the estimate is shown against it; where it is a direction or a limit
(information consistency, growth with *n*), the table says which.

In [ ]:
show_study("3b", smoke = smoke,
  tables = list(
    "3b.1 Predictive matching" = "predictive_matching",
    "3b.2 Information consistency" = "information_consistency",
    "3b.3-3b.5 Rates: slope of the median log Bayes factor against log n" = "rates",
    "3b.6 Consistency under the alternative: median log Bayes factor per n" = "alternative",
    "3b.6 Consistency under the alternative: slopes" = "alternative_slopes",
    "3b.7 Intrinsic consistency" = "intrinsic_consistency",
    "3b.8-3b.9 Growing k: slope of the median log Bayes factor against log n" = "growing_slopes"
  ),
  notes = list(function(tables) {
    divergence_sentence(tables$divergences, "bayes_factors", "with_divergences", "Bayes factors")
  })
)

**Result** (run of 19 November 2026, tag `v1-results`; to refresh after a new run): every criterion is met by all three priors, the rates at 0.94–1.06 times the theory. With
90 null predictors at *n* = 360 the median BF for the null is e^72 for flat partial *r* against
about 4 for flat *R*. For one small real effect, flat partial *r*'s BF is about half the others'
(median log BF₁₀ 0.41 against 1.07 and 1.26 at *n* = 200).

---

# Details

The tables not needed to see each result: every row, the counts beyond 2 and 3 MCSEs (what chance
gives: about 1 in 20 beyond 2, 1 in 370 beyond 3), and the diagnostics. In 2b the counts are
against references a Bayesian estimate need not meet (shrinkage is expected), so they describe.

In [ ]:
show_details("1a", smoke = smoke, tables = list(
  "Quantiles of the prior draws against their expected values" = "quantiles",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Iterations" = "iterations"
))
show_details("1b", smoke = smoke, tables = list(
  "Change in the log Bayes factor, per dataset" = "invariance",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference"
))
show_details("2a", smoke = smoke, tables = list(
  "Ranks, with the chi-square over 20 rank bins" = "ranks",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Divergent transitions of the SBC fits" = "divergences"
))
show_details("2b", smoke = smoke, tables = list(
  "Estimates against their reference, per cell" = "estimates",
  "Beyond 2 and 3 Monte Carlo SEs of the reference (descriptive)" = "beyond_reference",
  "Stan's divergent transitions" = "divergent"
))
show_details("3a", smoke = smoke, tables = list(
  "Logistic calibration line per prior and cell (descriptive)" = "calibration_line",
  "Calibration per bin" = "calibration_bins",
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Approximate errors and divergent transitions" = "diagnostics"
))
show_details("3b", smoke = smoke, tables = list(
  "Beyond 2 and 3 Monte Carlo SEs" = "beyond_reference",
  "Divergent transitions" = "divergences"
))